In [1]:
# Cài đặt Unsloth - Lưu ý chọn đúng phiên bản cho môi trường Kaggle
!pip install unsloth
# Cài đặt thêm các thư viện hỗ trợ xử lý hình ảnh và video cho Qwen2-VL
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import os
import json
from PIL import Image
import torch
from unsloth import FastVisionModel
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# ==========================================
# 1. ĐỊNH NGHĨA ĐƯỜNG DẪN TỪ DATASET MỚI
# ==========================================
# Trỏ đến file JSON (ở đây tôi giả định bạn muốn chạy trên file test.json)
JSON_PATH = "/kaggle/input/datasets/spixalo/trafic/traffic-signs-json/test.json" 

# Trỏ đến thư mục chứa ảnh
IMAGES_DIR = "/kaggle/input/datasets/maitam/vietnamese-traffic-signs/archive/images"

# Nơi lưu kết quả
OUTPUT_JSON_PATH = "./b1_predictions.json"

In [4]:
# ==========================================
# 2. KHỞI TẠO MÔ HÌNH (UNSLOTH 4-BIT)
# ==========================================
print("Đang đánh thức 'con quái vật' Qwen2-VL 7B (bản 4-bit)...")
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit = True,
)
FastVisionModel.for_inference(model) # Bật chế độ chạy nhanh

Đang đánh thức 'con quái vật' Qwen2-VL 7B (bản 4-bit)...
==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Qwen2VLForConditionalGeneration(
  (model): Qwen2VLModel(
    (visual): Qwen2VisionTransformerPretrainedModel(
      (patch_embed): PatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2VLVisionBlock(
          (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
          (attn): VisionAttention(
            (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
            (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): VisionMlp(
            (fc1): Linear4bit(in_features=1280, out_features=5120, bias=True)
            (act): QuickGELUActivation()
            (fc2): Linear4bit(in_features=5120, out_features=1280, bias=True)
          )
        )
      )
      (merger): PatchMerger(
      

In [5]:
# ==========================================
# 3. ĐỌC DỮ LIỆU TỪ FILE JSON
# ==========================================
print(f"\nĐang đọc dữ liệu từ: {JSON_PATH}")

# Sử dụng thư viện json thay vì pandas
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f"Tuyệt vời, hệ thống đã tìm thấy {len(dataset)} câu hỏi trong file JSON.")

results = []


Đang đọc dữ liệu từ: /kaggle/input/datasets/spixalo/trafic/traffic-signs-json/test.json
Tuyệt vời, hệ thống đã tìm thấy 260 câu hỏi trong file JSON.


In [6]:
# ==========================================
# 4. TIẾN HÀNH DỰ ĐOÁN
# ==========================================
pbar = tqdm(dataset, desc="Đang xử lý VQA")

for item in pbar:
    file_name = item['image_name']
    question = item['question']
    
    # Lấy đáp án từ mảng (Vì 'answer' đang là dạng list ví dụ: ["Biển báo..."])
    ground_truth = item['answer'][0] if len(item['answer']) > 0 else ""
    
    image_path = os.path.join(IMAGES_DIR, file_name)
    
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        tqdm.write(f"⚠️ Bỏ qua ảnh {file_name} do lỗi: {e}")
        continue
        
    prompt_text = f"{question} Trả lời dưới 11 từ."
    
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_text}
        ]}
    ]
    
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, input_text, return_tensors="pt").to("cuda")

    # Sinh câu trả lời
    output_ids = model.generate(**inputs, max_new_tokens=27, max_length=None)
    prediction_full = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    prediction = prediction_full.split("assistant\n")[-1].strip()

    # Cập nhật thông tin lên thanh tiến trình
    short_pred = prediction[:15] + "..." if len(prediction) > 15 else prediction
    pbar.set_postfix({"Ảnh": file_name, "Dự đoán": short_pred})

    results.append({
        "index": item['index'],
        "file_name": file_name,
        "question": question,
        "ground_truth": ground_truth,
        "prediction": prediction
    })

Đang xử lý VQA: 100%|██████████| 260/260 [11:36<00:00,  2.68s/it, Ảnh=1238.jpg, Dự đoán=11]


In [7]:
# ==========================================
# 5. LƯU KẾT QUẢ ĐỂ ĐÁNH GIÁ
# ==========================================
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\nHoàn thành xuất sắc! Toàn bộ kết quả dự đoán đã được lưu gọn gàng tại: {OUTPUT_JSON_PATH}")


Hoàn thành xuất sắc! Toàn bộ kết quả dự đoán đã được lưu gọn gàng tại: ./b1_predictions.json
